# 46 — Resume vs JD Matching
**Goal:** Build a complete resume-to-job-description matcher using multiple signals.

A resume and a JD describe the same person from two directions, but they almost never use the same words. A matcher that only checks "does the resume mention Python?" is brittle: it misses synonyms, paraphrases, and the *strength* of a match. This chapter builds a `ResumeJDMatcher` that combines four independent signals — skill overlap, experience, education, and embedding similarity — into one weighted score.

**Why it matters for resumes / ATS:** an ATS's core decision is "how good is this candidate for this role?". A single-signal answer (keyword hit or miss) is easy to game and hard to debug. Weighted multi-signal scoring produces a number you can decompose into *why*: which skills matched, whether the embedding agrees, and which bonus fired. That transparency is what lets a recruiter or hiring manager trust — and tune — the system.

![Resume vs JD Matching Pipeline](../../../assets/images/resume_jd_matching_1785491166825.png)

> **Figure:** The semantic matching pipeline — resume and JD are embedded independently, then compared via cosine similarity to produce a match score and skill gap analysis.

The pipeline has three stages. **Embed:** resume and JD texts are encoded into fixed-length vectors by a sentence-transformer model (`all-MiniLM-L6-v2`). **Compare:** cosine similarity measures how close the two vectors are — the geometric proxy for "talks about the same things". **Score:** the similarity is folded into a weighted total alongside keyword-level signals (skill overlap, experience, education) so the final number reflects both *semantic* closeness and *explicit* evidence.

## 1. Multi-Signal Matching Strategy

Any single signal can lie. Keyword overlap misses synonyms and typos; embedding similarity is blind to explicit evidence like "5+ years" or "Masters"; experience and education alone say nothing about the actual skills. The fix is to **combine several weak signals** with weights and let them corroborate each other — agreement between signals is what makes a score confident.

**What the code does:** the first cell prints the weighting scheme used by the rest of the chapter:

| Signal | Weight | What it measures |
|---|---|---|
| Skill overlap | 40% | exact + fuzzy keyword matches between resume and JD skills |
| Experience level | 20% | years-of-experience evidence in the resume text |
| Education | 15% | degree keywords (Masters / PhD / Bachelor) |
| Embedding similarity | 25% | semantic closeness of the two full texts |

**Try it:** the weights sum to 100% and the largest share goes to *skills* — the signal most specific to a job. The two smallest weights (experience, education) act as tie-breakers between candidates who are otherwise skill-equivalent.

In [ ]:
print('''Resume-JD matching uses multiple signals:
1. Skill overlap (exact + semantic) -> 40% weight
2. Experience level match -> 20% weight
3. Education match -> 15% weight
4. Embedding similarity -> 25% weight

Final score = weighted combination of all signals.
Confidence comes from agreement between signals.''')

## 2. Building the Matcher

The matcher is a single class, `ResumeJDMatcher`, that keeps the model and the skill vocabulary in one place and exposes three methods: `skill_overlap()`, `embedding_match()`, and `match()`. The class is defensive by design: if the embedding model fails to load, `has_model` flips to `False` and `embedding_match()` returns a neutral 0.5 instead of crashing.

**What the code does:**
- `skill_overlap()` uses `rapidfuzz.fuzz.partial_ratio` with a > 85 threshold, so a resume skill counts as matched when it is a close fuzzy variant of a JD skill — tolerant of typos and inflections.
- `embedding_match()` truncates both texts to 512 chars before encoding, so long resumes do not blow up encode time, and returns `util.cos_sim(emb1, emb2)`.
- `match()` extracts skills by substring lookup against `skills_db`, then combines `skill_score` (0.40x), `emb_score` (0.25x), and two regex bonuses: `\d\+?\s*years?` for experience and `(masters|phd|bachelor)` for education.

**Expected on the sample pair:** both regexes fire ("5+ years", "Masters"), and the resume's Python/NLP/TensorFlow are all required by the JD, so skill overlap is 3/3 and the score lands near the top of the range; the exact total depends on the embedding cosine term.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import re
from rapidfuzz import fuzz

class ResumeJDMatcher:
    def __init__(self):
        self.skills_db = ["Python", "TensorFlow", "PyTorch", "NLP", "SQL", "AWS", "Docker", "Spark", "Java", "React"]
        try:
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.has_model = True
        except:
            self.has_model = False
    
    def skill_overlap(self, resume_skills, jd_skills):
        if not resume_skills or not jd_skills: return 0
        matched = sum(1 for s in resume_skills if any(
            fuzz.partial_ratio(s.lower(), j.lower()) > 85 for j in jd_skills
        ))
        return matched / max(len(jd_skills), 1)
    
    def embedding_match(self, resume_text, jd_text):
        if not self.has_model: return 0.5
        emb1 = self.model.encode(resume_text[:512])
        emb2 = self.model.encode(jd_text[:512])
        return util.cos_sim(emb1, emb2).item()
    
    def match(self, resume_text, jd_text):
        # Extract skills from both
        resume_skills = [s for s in self.skills_db if s.lower() in resume_text.lower()]
        jd_skills = [s for s in self.skills_db if s.lower() in jd_text.lower()]
        
        skill_score = self.skill_overlap(resume_skills, jd_skills) * 0.40
        emb_score = self.embedding_match(resume_text, jd_text) * 0.25
        # Experience and education bonuses (simplified)
        exp_bonus = 0.20 if re.search(r"\\d\\+?\\s*years?", resume_text, re.IGNORECASE) else 0.0
        edu_bonus = 0.15 if re.search(r"(masters|phd|bachelor)", resume_text, re.IGNORECASE) else 0.0
        
        total = skill_score + emb_score + exp_bonus + edu_bonus
        return {
            "score": round(total, 3),
            "details": {"skill_match": skill_score, "embedding": emb_score,
                        "experience": exp_bonus, "education": edu_bonus},
            "resume_skills_found": resume_skills,
            "jd_skills_required": jd_skills,
        }

matcher = ResumeJDMatcher()
resume = """Senior Data Scientist with 5+ years Python, NLP, TensorFlow experience.
Masters in Computer Science."""
jd = """Senior Data Scientist required. Python, TensorFlow, NLP preferred.
5+ years experience. MS/PhD preferred."""

result = matcher.match(resume, jd)
print(f"Match score: {result['score']}")
print(f"Breakdown: {result['details']}")
print(f"Resume skills: {result['resume_skills_found']}")
print(f"JD requires: {result['jd_skills_required']}")

## 3. Testing on Multiple JDs

A matcher that only scores well on one hand-written pair proves nothing. The real test is the same resume run against **diverse JDs** — a data-science role it should match, and Java-backend, DevOps, and frontend roles it should mostly reject.

**What the code does:** loops over four JD strings, calls `matcher.match(resume, jd_text)` for each, and prints the score next to the JD text.

**Expected behavior:** JD 1 ("Data Scientist — Python, ML, NLP, TensorFlow") should score far above the others, because Python/NLP/TensorFlow appear on both sides — a 3/3 skill overlap worth the full 0.40. The Java-backend and frontend JDs share no skills with the resume, so their scores rest almost entirely on the embedding term plus whatever bonuses fire — a clear, interpretable gap.

**Why this matters:** the *relative ordering* across JDs is what a ranking system consumes. If the wrong JD wins, the `details` dict shows exactly which signal misled you.

In [ ]:
jds = [
    "Data Scientist — Python, ML, NLP, TensorFlow required",
    "Java Backend Developer — Spring, Microservices, AWS",
    "DevOps Engineer — Docker, Kubernetes, CI/CD",
    "Frontend Developer — React, TypeScript, CSS",
]
for i, jd_text in enumerate(jds):
    r = matcher.match(resume, jd_text)
    print(f"  JD {i+1}: {r['score']:.3f} — {jd_text}")

## Summary: Multi-signal matching with weighted scoring. Transparent, debuggable, no black box.

The chapter assembled a working matcher from four signals: fuzzy skill overlap (40%), embedding similarity (25%), experience (20%), and education (15%). Because every component is a plain function returning a number you can print, a low score is never a mystery — `match()` returns a `details` dict with each signal's contribution and the exact skill lists found on both sides. That decomposability is the difference between a scoring *tool* and a scoring *black box*, and it is what makes the approach safe to put in front of recruiters.

## Key Insight

**Match scores must be decomposable — a number without a breakdown is a guess.**

A weighted blend of skill overlap, experience, education, and embedding similarity is only useful because each signal is inspectable in isolation. When scores misrank candidates, the `details` breakdown tells you which signal was wrong and how to re-weight it. This transparency is the foundation for the rest of the block: vector search (47–48) accelerates how the embedding signal is computed at scale, and the benchmark (49) measures whether that embedding signal is trustworthy. All of it converges in Ch. 50 — ATS Rule Design.